In [1]:
import jax
import numpy as np
import jax.numpy as jnp
import jax.experimental.pallas as pl
import timeit

from functools import partial

def add_vectors_kernel(x_ref, y_ref, o_ref):
  x, y = x_ref[...], y_ref[...]
  o_ref[...] = x + y

@jax.jit
def add_vectors(x: jax.Array, y: jax.Array) -> jax.Array:
  return pl.pallas_call(
      add_vectors_kernel,
      out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype)
  )(x, y)
add_vectors(jnp.arange(8), jnp.arange(8))

Array([ 0,  2,  4,  6,  8, 10, 12, 14], dtype=int32)

In [2]:
def iota_kernel(o_ref):
  i = pl.program_id(0)
  o_ref[i] = i

def iota(size: int):
  return pl.pallas_call(iota_kernel,
                        out_shape=jax.ShapeDtypeStruct((size,), jnp.int32),
                        grid=(size,))()
iota(8)

Array([0, 1, 2, 3, 4, 5, 6, 7], dtype=int32)

In [3]:
def bitonic_sort_kernel(a_ref, out_ref):
    a = a_ref[...]
    out = out_ref[...]
    n = a.shape[0]
    idx = pl.program_id(0)
    
    def swap(a, idx, ixj, dir):
        cond = (a[idx] > a[ixj]) if dir else (a[idx] < a[ixj])
        temp = jnp.where(cond, a[ixj], a[idx])
        a = a.at[idx].set(jnp.where(cond, a[idx], a[ixj]))
        a = a.at[ixj].set(temp)
        return a

    size = 2
    while size <= n:
        dir = ((idx & size) == 0)
        stride = size // 2
        while stride > 0:
            ixj = idx ^ stride
            if ixj > idx:
                a = swap(a, idx, ixj, dir)
            stride //= 2
        size *= 2

def bitonic_sort(a):
    n = a.shape[0]
    assert (n & (n - 1) == 0), "배열의 크기는 2의 거듭제곱이어야 합니다."
    sorted_a = pl.pallas_call(bitonic_sort_kernel, out_shape=jax.ShapeDtypeStruct(a.shape, a.dtype), grid=(n,))(a)
    return sorted_a

a = jnp.array([3, 7, 4, 8, 6, 2, 1, 5])
print(bitonic_sort(a))

TypeError: bitonic_sort_kernel() takes 1 positional argument but 2 were given